<a href="https://colab.research.google.com/github/RaihahMahmud/FlyRank-AI--starter-ML-Internship-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

## Data Contract

**Unit of analysis**
One row represents one content page for one client on one reporting date.

**Table used**
fact_content_daily_performance

**Time window**
March 1, 2026 – March 31, 2026

**Prediction / ranking goal**
Rank content pages by priority for review as part of the Refresh / Content Opportunity Scoring lane.

**Deliberately excluded**
Client identifiers and content identifiers are excluded as model features because they are pseudonymous identifiers used only for grouping and splitting data, not for prediction.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


In [23]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [25]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061,413966


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

For this project, the unit of analysis is one content page on one reporting day.

Each row in the `fact_content_daily_performance` table represents the performance of a single pseudonymized content page for one client on a specific report date.

For this assignment, I use data from **March 2026** (`month = '2026-03'`). This is a mid-panel month recommended for exploration because it avoids using the final month as a development period.

The goal of my lane is to build a ranking system that helps prioritize which content pages should be reviewed by content editors.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: Feature / Label / Context / Excluded

### Features
These are variables that may help estimate which content pages deserve review.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_pageviews

### Label / Proxy

For my lane, the objective is to rank pages by review priority rather than predict a predefined label.

The priority score will act as a proxy for identifying pages that may deserve review based on historical search and engagement signals.

### Context

These columns provide useful information but will not be used directly as predictive features.

- report_date
- client_hash_id
- content_hash_id
- month

### Excluded

Some fields are intentionally excluded.

- client_hash_id — identifier only, not meaningful for prediction.
- content_hash_id — identifier only.
- report_date — used for filtering and validation, not as a feature.
- Future or label-derived information will be excluded to avoid data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Query 1 — Verify the Grain

This query checks whether each row represents one unique combination of report date, client, and content page.

In [29]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


## Verification Query 2 – Verify the Time Window

This query confirms that the selected data only contains records from the March 2026 observation window and reports the first and last available dates.

In [30]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## Verification Query 3 – Verify Data Availability

This query checks how many rows have usable Google Search Console (GSC) and Google Analytics 4 (GA4) data. Only rows where the availability flags are TRUE should be considered for analyses that rely on those sources.

In [31]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

,total_rows,gsc_available,ga4_available
0,9841378,3611061,413966


The three verification queries confirm that:

- Each row represents one unique content page for one client on one reporting date.
- The selected data covers the full month of March 2026.
- Search Console and GA4 data are only available for a subset of rows, so future analyses should filter using the availability flags.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset has several limitations that should be considered during analysis.

Some clients have much shorter historical data than others, so comparisons across clients may not always be fair.
Some rows contain Google Search Console data but do not yet have Google Analytics data available, so engagement-related features may be missing for those records.
This analysis is based on historical observations and cannot prove that refreshing content will improve future performance.
To reduce the risk of data leakage, future information or label-derived fields will not be used as model features.

This is honest, aligns with the internship guidance, and uses the recommended careful wording.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.